In [81]:
from spacerocks import SpaceRock
from spacerocks.time import Time
from spacerocks.observing import Observatory, Observation
from spacerocks.spice import SpiceKernel
from spacerocks.nbody import Simulation, Force
import numpy as np


import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

kernel = SpiceKernel()
kernel.load("/Users/kjnapier/data/spice/latest_leapseconds.tls")
kernel.load("/Users/kjnapier/data/spice/sb441-n16.bsp")
kernel.load("/Users/kjnapier/data/spice/de441_part-1.bsp")
kernel.load("/Users/kjnapier/data/spice/de441_part-2.bsp")
kernel.load("/Users/kjnapier/data/spice/earth_1962_240827_2124_combined.bpc")

Loading kernel: /Users/kjnapier/data/spice/latest_leapseconds.tls
Loading kernel: /Users/kjnapier/data/spice/sb441-n16.bsp
Loading kernel: /Users/kjnapier/data/spice/de441_part-1.bsp
Loading kernel: /Users/kjnapier/data/spice/de441_part-2.bsp
Loading kernel: /Users/kjnapier/data/spice/earth_1962_240827_2124_combined.bpc


In [82]:
w84 = Observatory.from_obscode('w84')

In [93]:
epoch = Time.now()


rock = SpaceRock.from_horizons("Arrokoth", epoch=epoch, origin="ssb", reference_plane="J2000")
sim = Simulation.horizons(epoch, "J2000", "ssb")
sim.add(rock)

In [96]:
rock.r

43.09732054016736

In [84]:
observations = []    
for idx in range(0, 300, 100):
    sim.integrate(epoch + idx)
    observer = w84.at(epoch + idx, reference_plane="J2000", origin="ssb")
    rock = sim.get_particle("Arrokoth")
    obs = rock.observe(observer)
    observations.append(obs)

In [92]:
[obs.epoch for obs in observations]

[Time: 2460713.4646549183 TDB JD,
 Time: 2460813.4646549216 TDB JD,
 Time: 2460913.4646548927 TDB JD]

In [85]:
smear = 0.0 #1/3600 * np.pi / 180

In [86]:
simulated_observations = []
for obs in observations:
    ra = obs.ra
    dec = obs.dec
    #ra += np.random.normal(0, smear)
    #dec += np.random.normal(0, smear)
    epoch = obs.epoch
    observer = obs.observer
    cov = [[smear**2, 0], [0, smear**2]]
    simulated_o = Observation.from_astrometry(obs.epoch, ra, dec, obs.observer)
    #simulated_o.set_covariance(cov)
    simulated_observations.append(simulated_o)

In [91]:
simulated_observations[0].pointing

AttributeError: 'builtins.Observation' object has no attribute 'pointing'

In [71]:
simulated_observations

Observation:
  ra: 5.152508540464163
  dec: -0.34373437631876697
  ra_rate: None
  dec_rate: None
  range: None
  range_rate: None
  epoch: Time { epoch: 2460978.4578493685, timescale: TDB, format: JD }
  observer: Observer { spacerock: SpaceRock { name: "earth", epoch: Time { epoch: 2460978.4578493685, timescale: TDB, format: JD }, reference_plane: J2000, origin: SSB, position: [[0.795636874595605, 0.5358908784598652, 0.23243946316075945]], velocity: [[-0.010318853651529587, 0.012793122408275013, 0.00547714631605668]], properties: Some(Properties { mass: Some(3.0034896154502038e-6), absolute_magnitude: None, gslope: None, radius: None, albedo: None }) }, observatory: GroundObservatory { obscode: "W84", lon: 5.047380146629623, lat: -0.5236462337787118, rho: 0.9995038419300848 } }
Observation:
  ra: 5.191502052544712
  dec: -0.3378132829598074
  ra_rate: None
  dec_rate: None
  range: None
  range_rate: None
  epoch: Time { epoch: 2461078.4578493685, timescale: TDB, format: JD }
  obser

[, , ]

In [72]:
from spacerocks.orbfit import gauss

In [74]:
rocks = gauss(simulated_observations[0], simulated_observations[100], simulated_observations[200], min_distance=1e-6)

IndexError: list index out of range

In [80]:
for rock in rocks:
    print(rock.a(), rock.e(), rock.inc())

-43.97095080566391 43.18115224162581 2.772960853939401
-39.00449516998293 48.47504720873672 2.7723604518572236
772.0118443497529 0.9721327890928131 2.7648342878523664


In [76]:
rocks[0].a()

-43.97095080566391